# HVAC v19 retrain — SMALL bundle (241 MB) via Drive link

Backup path: uses the **241 MB** bundle `hvac_v19xs_tiled_colab.zip` (uploads to Drive ~4x
faster than the 940 MB one) and downloads it into Colab with `gdown` (no upload widget, no phone).

### Setup
1. In Drive, upload **`hvac_v19xs_tiled_colab.zip`** (~241 MB) and wait for the green check.
2. Right-click it -> **Share -> Anyone with the link -> Copy link.**
3. Paste the link into `LINK` in cell 3.
4. `Runtime -> Change runtime type -> GPU`, then `Runtime -> Run all`.

Produces `hvac_yolov8s_v19.pt` -> drop it in the repo `models/` folder; the gate runs automatically.

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE -> Runtime > Change runtime type > GPU, then Run all again')

In [ ]:
!pip -q install ultralytics==8.4.51 gdown

In [ ]:
# Paste your Drive 'Anyone with the link' URL for hvac_v19xs_tiled_colab.zip:
LINK = 'PASTE_DRIVE_SHARE_LINK_HERE'
import gdown, zipfile, os
assert LINK.startswith('http'), 'Paste the Drive share link into LINK first.'
gdown.download(url=LINK, output='bundle.zip', fuzzy=True, quiet=False)
sz = os.path.getsize('bundle.zip') / 1e6
print(f'downloaded bundle.zip = {sz:.0f} MB')
assert sz > 150, 'Too small - Drive returned an HTML page, not the zip. Set sharing to "Anyone with the link".'
os.makedirs('/content/hvac', exist_ok=True)
with zipfile.ZipFile('bundle.zip') as z:
    z.extractall('/content/hvac')
os.chdir('/content/hvac')
print('ready:', os.getcwd(), sorted(os.listdir('.'))[:8])

In [ ]:
# Train. ~30-45 min on a T4. (Lower --epochs to 30 if you want it even faster.)
!python train_v11.py --data-yaml yolo_dataset_v19xs_tiled/data.yaml \
    --epochs 60 --device 0 --batch 16 --imgsz 640 --run-name v19

In [ ]:
import glob
from google.colab import files
cands = glob.glob('models/hvac_yolov8s_v19.pt') + glob.glob('runs/**/weights/best.pt', recursive=True)
assert cands, 'No weights found - check the training cell output.'
print('downloading', cands[0])
files.download(cands[0])